## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    CountVectorizer,
    IDF,
    StringIndexer,
    ChiSqSelector,
    Normalizer
)
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

## Global Variables

In [ ]:
INPUT_PATH = "hdfs:///dic_shared/amazon-reviews/full/reviews_devset.json"

FEATURE_OPTIONS = [2000, 500]
REG_PARAMS = [0.01, 0.1, 1.0]
STANDARDIZATION_OPTIONS = [True, False]
MAX_ITER_OPTIONS = [50, 100]

SEED = 42

## Environment Setup

In [5]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Part3_Text_Classification")
    .getOrCreate()
)

reviews = spark.read.json(INPUT_PATH)

## Dataset Preparation

In [6]:
df = reviews.select(
    col("category"),
    col("reviewText")
).na.drop(subset=["category", "reviewText"])

print("Number of reviews:", df.count())
print("Number of categories:", df.select("category").distinct().count())

Number of reviews: 78829
Number of categories: 22


## Train / Validation / Test Split

In [7]:
train_df, validation_df, test_df = df.randomSplit(
    [0.6, 0.2, 0.2],
    seed=SEED
)

print("Train size:", train_df.count())
print("Validation size:", validation_df.count())
print("Test size:", test_df.count())

Train size: 47552
Validation size: 15685
Test size: 15592


## Tokenization

In [8]:
token_split_re = r"[\s\d\(\)\[\]\{\}\.!?;,:+\-_\"'`~#@&\*%€\$\\/]+"

tokenizer = RegexTokenizer(
    inputCol="reviewText",
    outputCol="tokens",
    pattern=token_split_re,
    gaps=True,
    toLowercase=True,
    minTokenLength=1
)

## Stop Words

In [9]:
remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

## Vectorizer

In [10]:
count_vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="tf_features",
    minDF=1.0
)

## TF-IDF

In [11]:
idf = IDF(
    inputCol="tf_features",
    outputCol="tfidf_features"
)

## Label Indexer

In [12]:
label_indexer = StringIndexer(
    inputCol="category",
    outputCol="label"
)

## Normalizer

In [13]:
normalizer = Normalizer(
    inputCol="selected_features",
    outputCol="normalized_features",
    p=2.0
)

## Evaluator

In [14]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

## Grid Search

In [15]:
results = []

for num_features in FEATURE_OPTIONS:
    
    print(f"\n=== Testing numTopFeatures = {num_features} ===")
    
    # Chi-Square Selector
    chi_sq_selector = ChiSqSelector(
        numTopFeatures=num_features,
        featuresCol="tfidf_features",
        outputCol="selected_features",
        labelCol="label"
    )
    
    for reg_param in REG_PARAMS:
        for standardization in STANDARDIZATION_OPTIONS:
            for max_iter in MAX_ITER_OPTIONS:
                
                print(
                    f"Training model with "
                    f"regParam={reg_param}, "
                    f"standardization={standardization}, "
                    f"maxIter={max_iter}"
                )
                
                # Binary SVM
                lsvc = LinearSVC(
                    featuresCol="normalized_features",
                    labelCol="label",
                    regParam=reg_param,
                    standardization=standardization,
                    maxIter=max_iter
                )
                
                # Multi-class Strategy
                ovr = OneVsRest(
                    classifier=lsvc,
                    labelCol="label",
                    featuresCol="normalized_features"
                )
                
                # Pipeline
                pipeline = Pipeline(stages=[
                    tokenizer,
                    remover,
                    count_vectorizer,
                    idf,
                    label_indexer,
                    chi_sq_selector,
                    normalizer,
                    ovr
                ])
                
                # Train model
                model = pipeline.fit(train_df)
                
                # Validation predictions
                validation_predictions = model.transform(validation_df)
                
                # Validation F1
                validation_f1 = evaluator.evaluate(validation_predictions)
                
                # Store result
                results.append({
                    "numTopFeatures": num_features,
                    "regParam": reg_param,
                    "standardization": standardization,
                    "maxIter": max_iter,
                    "validation_f1": validation_f1,
                    "model": model
                })
                
                print(f"Validation F1: {validation_f1:.4f}")


=== Testing numTopFeatures = 2000 ===
Training model with regParam=0.01, standardization=True, maxIter=50
Validation F1: 0.6629
Training model with regParam=0.01, standardization=True, maxIter=100
Validation F1: 0.6629
Training model with regParam=0.01, standardization=False, maxIter=50
Validation F1: 0.5560
Training model with regParam=0.01, standardization=False, maxIter=100
Validation F1: 0.5557
Training model with regParam=0.1, standardization=True, maxIter=50
Validation F1: 0.6569
Training model with regParam=0.1, standardization=True, maxIter=100
Validation F1: 0.6558
Training model with regParam=0.1, standardization=False, maxIter=50
Validation F1: 0.5541
Training model with regParam=0.1, standardization=False, maxIter=100
Validation F1: 0.5534
Training model with regParam=1.0, standardization=True, maxIter=50
Validation F1: 0.6202
Training model with regParam=1.0, standardization=True, maxIter=100
Validation F1: 0.6199
Training model with regParam=1.0, standardization=False, m

## Best Model Selection

In [16]:
best_result = max(results, key=lambda x: x["validation_f1"])

print("\n==============================")
print("BEST MODEL")
print("==============================")
print("numTopFeatures:", best_result["numTopFeatures"])
print("regParam:", best_result["regParam"])
print("standardization:", best_result["standardization"])
print("maxIter:", best_result["maxIter"])
print("Validation F1:", best_result["validation_f1"])


BEST MODEL
numTopFeatures: 2000
regParam: 0.01
standardization: True
maxIter: 100
Validation F1: 0.6629267517409745


# Final Test Evaluation

In [17]:
best_model = best_result["model"]

test_predictions = best_model.transform(test_df)

test_f1 = evaluator.evaluate(test_predictions)

print("\n==============================")
print("FINAL TEST RESULT")
print("==============================")
print("Test F1 Score:", test_f1)


FINAL TEST RESULT
Test F1 Score: 0.6600370415883677


## Display All Results Sorted by Validation F1

In [18]:
sorted_results = sorted(
    results,
    key=lambda x: x["validation_f1"],
    reverse=True
)

print("\n==============================")
print("ALL RESULTS")
print("==============================")

for result in sorted_results:
    print(
        f"Features={result['numTopFeatures']}, "
        f"regParam={result['regParam']}, "
        f"standardization={result['standardization']}, "
        f"maxIter={result['maxIter']}, "
        f"Validation F1={result['validation_f1']:.4f}"
    )


ALL RESULTS
Features=2000, regParam=0.01, standardization=True, maxIter=100, Validation F1=0.6629
Features=2000, regParam=0.01, standardization=True, maxIter=50, Validation F1=0.6629
Features=2000, regParam=0.1, standardization=True, maxIter=50, Validation F1=0.6569
Features=2000, regParam=0.1, standardization=True, maxIter=100, Validation F1=0.6558
Features=2000, regParam=1.0, standardization=True, maxIter=50, Validation F1=0.6202
Features=2000, regParam=1.0, standardization=True, maxIter=100, Validation F1=0.6199
Features=500, regParam=0.01, standardization=True, maxIter=50, Validation F1=0.5561
Features=2000, regParam=0.01, standardization=False, maxIter=50, Validation F1=0.5560
Features=2000, regParam=0.01, standardization=False, maxIter=100, Validation F1=0.5557
Features=2000, regParam=1.0, standardization=False, maxIter=50, Validation F1=0.5550
Features=500, regParam=0.01, standardization=True, maxIter=100, Validation F1=0.5543
Features=2000, regParam=0.1, standardization=False,

## Save Results to File

In [19]:
OUTPUT_PATH = "../result/output_part3_results.txt"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write("Best Model Parameters\n")
    f.write("=====================\n")
    f.write(f"numTopFeatures: {best_result['numTopFeatures']}\n")
    f.write(f"regParam: {best_result['regParam']}\n")
    f.write(f"standardization: {best_result['standardization']}\n")
    f.write(f"maxIter: {best_result['maxIter']}\n")
    f.write(f"Validation F1: {best_result['validation_f1']:.6f}\n")
    f.write(f"Test F1: {test_f1:.6f}\n\n")

    f.write("All Results\n")
    f.write("===========\n")

    for result in sorted_results:
        f.write(
            f"Features={result['numTopFeatures']}, "
            f"regParam={result['regParam']}, "
            f"standardization={result['standardization']}, "
            f"maxIter={result['maxIter']}, "
            f"Validation F1={result['validation_f1']:.6f}\n"
        )

print("Results written to:", OUTPUT_PATH)

Results written to: ../result/output_part3_results.txt


## Stop Spark

In [ ]:
spark.stop()